# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [1]:
!pip install langchain_community langchain_experimental langchain_openai

INFO: pip is looking at multiple versions of langchain-experimental to determine which version is compatible with other requirements. This could take a while.


In [4]:
!pip install google-search-results > /dev/null

In [5]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 20.4 MB/s  0:00:006m-:--:--


In [1]:
from typing import Optional

from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_experimental.autonomous_agents import BabyAGI
from langchain_openai import OpenAI, OpenAIEmbeddings

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [2]:
# %pip install faiss-cpu > /dev/null
# %pip install google-search-results > /dev/null
from langchain.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS

In [3]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
SERPAPI_API_KEY = os.getenv('SERPAPI_API_KEY')

In [4]:
from langchain_community.utilities import SerpAPIWrapper

search = SerpAPIWrapper()
response = search.run("latest AI regulation news in the European Union")

response


'[\'The European Union (EU) is ushering in a new era of artificial intelligence (AI) regulation with the introduction of the AI act dated 13 ...\', \'The AI Act is a European regulation on artificial intelligence (AI) – the first comprehensive regulation on AI by a major regulator anywhere.\', \'The AI Act is the first-ever legal framework on AI, which addresses the risks of AI and positions Europe to play a leading role globally.\', \'The EuroHPC regulation was amended in 2024 to include the development and operation of AI Factories —dynamic ecosystems that promote innovation ...\', \'On October 10, 2025, the National AI Law entered into force. Unlike the EU AI Act, which provides for gradual implementation and will only ...\', "September–October 2025 saw a surge of EU-level initiatives aimed at strengthening Europe\'s AI governance, innovation, and digital infrastructure.", "The EU\'s world-first AI regulation is also facing several changes, notably the timelines for entry into appli

In [6]:
# Define your embedding model
embeddings_model = OpenAIEmbeddings()
# Initialize the vectorstore as empty
import faiss

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model.embed_query, index, InMemoryDocstore({}), {})

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [8]:
from langchain.agents import AgentExecutor, Tool, ZeroShotAgent
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_community.utilities import SerpAPIWrapper
from langchain_openai import OpenAI

# -------- Planner (Task Creation Chain) --------

todo_prompt = PromptTemplate.from_template(
    """
You are a senior research planner AI.
Break the following objective into a small number of high-impact, concrete research tasks.
Each task must be necessary and logically ordered.

Objective: {objective}

Return the tasks as a numbered list.
"""
)

planner_llm = OpenAI(temperature=0)
todo_chain = LLMChain(llm=planner_llm, prompt=todo_prompt)

# -------- Tools --------

search = SerpAPIWrapper()

tools = [
    Tool(
        name="Search",
        func=search.run,
        description="Use this tool to find up-to-date, factual information from the web, especially about laws, regulations, and recent events.",
    )
]

# -------- Agent Prompt --------

prefix = """
You are an autonomous research agent.
Your job is to execute ONE task at a time to help achieve the overall objective:

{objective}

You must:
- Use tools when factual or up-to-date information is required
- Be concise and factual
- Produce outputs that can be reused by future tasks

Previously completed tasks:
{context}
"""

suffix = """
Current task:
{task}

{agent_scratchpad}
"""

prompt = ZeroShotAgent.create_prompt(
    tools,
    prefix=prefix,
    suffix=suffix,
    input_variables=["objective", "task", "context", "agent_scratchpad"],
)


In [9]:
llm = OpenAI(temperature=0)
llm_chain = LLMChain(llm=llm, prompt=prompt)
tool_names = [tool.name for tool in tools]
agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent, tools=tools, verbose=True
)

/var/folders/jk/_xp09qcs57sdz1r74tjmbdpr0000gn/T/ipykernel_68599/3064747660.py:4: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)


### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [10]:
OBJECTIVE = "Create a concise market intelligence brief about the current state of AI regulation in the European Union"


In [11]:
# Logging of LLMChains
verbose = False
# If None, will keep on going forever
max_iterations: Optional[int] = 3
baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    task_execution_chain=agent_executor,
    verbose=verbose,
    max_iterations=max_iterations,
)

In [12]:
baby_agi({"objective": OBJECTIVE})

/var/folders/jk/_xp09qcs57sdz1r74tjmbdpr0000gn/T/ipykernel_68599/3867971396.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  baby_agi({"objective": OBJECTIVE})



*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


> Entering new AgentExecutor chain...
Question: What is the current state of AI regulation in the European Union?
Thought: I should use the Search tool to find up-to-date, factual information about AI regulation in the European Union.
Action: Search
Action Input: "AI regulation European Union"
Observation: ['In April 2021, the European Commission proposed the first EU artificial intelligence law, establishing a risk-based AI classification system. AI ...', 'The AI Act is a European regulation on artificial intelligence (AI) – the first comprehensive regulation on AI by a major regulator anywhere.', 'The AI Act (Regulation (EU) 2024/1689 laying down harmonised rules on artificial intelligence) is the first-ever comprehensive legal framework on AI worldwide.', 'It establishes a common regulatory and legal framework for AI within the European Union (EU). The regulation entered into force on 1 August 2024

{'objective': 'Create a concise market intelligence brief about the current state of AI regulation in the European Union'}